# DL Streamer: car detection and color classification (teaching notebook)

This notebook builds **GStreamer / DL Streamer** pipelines step by step. Each section defines paths **next to** the pipeline that uses them, shows the **full pipeline string** in the cell, and runs it with **`run_pipeline`** (subprocess only — no hidden construction).

Adjust file paths to match your machine. If `decodebin3` is unavailable, set `GST_DECODEBIN=decodebin` in the environment or replace `decodebin3` with `decodebin` in the strings below.


## 1. Imports

`gst_path` only formats paths so `gst-launch-1.0` parses `location=` and `model=` correctly on Windows and Linux. `run_pipeline` runs a pipeline string and prints stderr on failure.


In [ ]:
import os
import platform
import subprocess
from pathlib import Path


def gst_path(path: str) -> str:
    """Format a filesystem path for GstParse (gst-launch)."""
    p = Path(path).expanduser()
    try:
        p = p.resolve()
    except OSError:
        p = Path(path).expanduser()
    s = p.as_posix()
    if not s.strip():
        return '""'
    if platform.system() == "Windows":
        esc = s.replace('"', '\\"')
        return f'"{esc}"'
    if any(c in s for c in ' "\'!=!') or " " in s:
        esc = s.replace('"', '\\"')
        return f'"{esc}"'
    return s


def run_pipeline(pipeline: str) -> None:
    """Execute a pipeline string with gst-launch-1.0 (does not build the string)."""
    pipeline = pipeline.strip()
    if not pipeline:
        raise ValueError("Empty pipeline string.")
    env = {**os.environ, "GST_DEBUG": "0", "GST_DEBUG_NO_COLOR": "1"}
    r = subprocess.run(
        ["gst-launch-1.0", pipeline],
        env=env,
        check=False,
        stderr=subprocess.PIPE,
        text=True,
    )
    if r.returncode != 0:
        if r.stderr and r.stderr.strip():
            print("--- gst-launch stderr ---")
            print(r.stderr.strip()[-8000:])
        print(f"(gst-launch exited with code {r.returncode}; closing the video window often causes nonzero.)")
    print("Pipeline ended.")


## 2. Display video (no inference)

Playback only: **decode** and **display**. No DL Streamer inference elements.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> C[videoconvert]
  C --> S[autovideosink]
```


In [ ]:
video_path = "videos/cars.mp4"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={gst_path(video_path)} !
{decodebin_element} !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 3. Validate DL Streamer installation

Confirm **GStreamer** can see DL Streamer plugins (here: `gvadetect`). If this fails, fix `GST_PLUGIN_PATH` / your DL Streamer install before running inference pipelines.


In [ ]:
r = subprocess.run(["gst-inspect-1.0", "gvadetect"], capture_output=True)
if r.returncode == 0:
    print("DL Streamer is installed and available (gvadetect plugin found).")
else:
    print("ERROR: gvadetect not found. Install DL Streamer and check GST_PLUGIN_PATH.")


## 4. Detection pipeline

`gvadetect` runs the detector IR; **`gvafpscounter`** is a separate FPS overlay element. **`gvawatermark`** draws metadata on the frame.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> Det["gvadetect<br/>(Detection)"]
  Det --> W[gvawatermark]
  W --> FPS["gvafpscounter<br/>(FPS)"]
  FPS --> C[videoconvert]
  C --> S[autovideosink]

  style Det fill:#ffcccc,stroke:#d32f2f,stroke-width:3px
  style FPS fill:#ffcccc,stroke:#d32f2f,stroke-width:3px
```


In [ ]:
video_path = "videos/cars.mp4"
detection_model_path = "model/detection.xml"
detection_device = "GPU"

print("Detection model:", detection_model_path)
print("Device:", detection_device)

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={gst_path(video_path)} !
{decodebin_element} !
gvadetect model={gst_path(detection_model_path)} device={detection_device} pre-process-backend=opencv !
gvawatermark !
gvafpscounter !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 5. Detection + classification pipeline

After detection, **`gvatrack`** associates ROIs across frames; **`gvaclassify`** runs the color (or attribute) classifier. `reclassify-interval` controls how often classification runs.


```mermaid
flowchart LR
  V[Video file] --> F[filesrc]
  F --> D[decodebin3]
  D --> Det[gvadetect]
  Det --> T[gvatrack]
  T --> Cls[gvaclassify]
  Cls --> Q[queue]
  Q --> W[gvawatermark]
  W --> FPS[gvafpscounter]
  FPS --> C[videoconvert]
  C --> S[autovideosink]
```


In [ ]:
video_path = "videos/cars.mp4"
detection_model_path = "model/detection.xml"
detection_device = "GPU"
classification_model_path = "model/classification.xml"
classification_device = "CPU"
reclassify_interval = 2

print("Detection model:", detection_model_path)
print("Detection device:", detection_device)
print("Classification model:", classification_model_path)
print("Classification device:", classification_device)

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline = f"""
filesrc location={gst_path(video_path)} !
{decodebin_element} !
gvadetect model={gst_path(detection_model_path)} device={detection_device} pre-process-backend=opencv !
gvatrack !
gvaclassify model={gst_path(classification_model_path)} device={classification_device} pre-process-backend=opencv reclassify-interval={reclassify_interval} !
queue ! gvawatermark ! gvafpscounter !
videoconvert !
autovideosink sync=true
"""
pipeline = " ".join(line.strip() for line in pipeline.splitlines() if line.strip())

print(pipeline)
run_pipeline(pipeline)


## 6. Benchmarking: one pipeline vs duplicated components

**Scaling** here means **repeating** processing blocks in the same `gst-launch` line (e.g. a second `gvadetect` after `queue`). There is no hidden helper — compare `pipeline_1` and `pipeline_2` directly.

Below, **`fakesink`** avoids opening a window; use `autovideosink` if you want to watch both stages during debugging.


```mermaid
flowchart LR
  subgraph one [pipeline_1]
    A1[filesrc] --> D1[decode] --> G1[gvadetect] --> F1[FPS] --> K1[fakesink]
  end
  subgraph two [pipeline_2]
    A2[filesrc] --> D2[decode] --> G2a[gvadetect] --> Q[queue] --> G2b[gvadetect] --> F2[FPS] --> K2[fakesink]
  end
```


In [ ]:
video_path = "videos/cars.mp4"
detection_model_path = "model/detection.xml"
detection_device = "GPU"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline_1 = f"""
filesrc location={gst_path(video_path)} !
{decodebin_element} !
gvadetect model={gst_path(detection_model_path)} device={detection_device} pre-process-backend=opencv !
gvafpscounter !
videoconvert !
fakesink sync=false
"""
pipeline_1 = " ".join(line.strip() for line in pipeline_1.splitlines() if line.strip())

print("--- pipeline_1 (single gvadetect) ---")
print(pipeline_1)
run_pipeline(pipeline_1)


In [ ]:
video_path = "videos/cars.mp4"
detection_model_path = "model/detection.xml"
detection_device = "GPU"

decodebin_element = os.environ.get("GST_DECODEBIN", "decodebin3")

pipeline_2 = f"""
filesrc location={gst_path(video_path)} !
{decodebin_element} !
gvadetect model={gst_path(detection_model_path)} device={detection_device} pre-process-backend=opencv !
queue !
gvadetect model={gst_path(detection_model_path)} device={detection_device} pre-process-backend=opencv !
gvafpscounter !
videoconvert !
fakesink sync=false
"""
pipeline_2 = " ".join(line.strip() for line in pipeline_2.splitlines() if line.strip())

print("--- pipeline_2 (two gvadetect blocks — duplicated component) ---")
print(pipeline_2)
run_pipeline(pipeline_2)
